# Fine-tune Amharic NER

In [ ]:
# 1) GPU + install (Colab already has PyTorch — do not pip-install torch)
import torch
print("cuda:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    print("WARNING: Runtime → Change runtime type → GPU")

!pip -q install -U transformers datasets seqeval accelerate evaluate scikit-learn

cuda: True Tesla T4


In [ ]:
from pathlib import Path

# 2) Target the uploaded amharic_ner.conll
CONLL_PATH = Path("amharic_ner.conll")

if not CONLL_PATH.exists():
    raise FileNotFoundError("amharic_ner.conll was not found in the Colab file explorer. Please drag and drop it into the left sidebar.")

print("Using", CONLL_PATH, "size", CONLL_PATH.stat().st_size)

Using amharic_ner.conll size 68561


In [ ]:
# 3) CoNLL load + subword label alignment
from pathlib import Path
from datasets import Dataset, DatasetDict

LABEL_LIST = [
    "O", "B-Product", "I-Product", "B-LOC", "I-LOC", "B-PRICE", "I-PRICE",
]
LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}


def read_conll(path: Path):
    sentences, tokens, labels = [], [], []
    for line in Path(path).read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            if tokens:
                sentences.append((tokens, labels))
                tokens, labels = [], []
            continue
        if line.startswith("#"):
            continue
        parts = line.split()
        if len(parts) >= 2:
            tokens.append(parts[0])
            labels.append(parts[-1])
    if tokens:
        sentences.append((tokens, labels))
    return sentences


def load_ner_dataset(conll_path, test_size=0.3, seed=42):
    records = []
    unknown = set()
    for tokens, tags in read_conll(conll_path):
        ids = []
        for tag in tags:
            if tag not in LABEL2ID:
                unknown.add(tag)
                ids.append(LABEL2ID["O"])
            else:
                ids.append(LABEL2ID[tag])
        records.append({"tokens": tokens, "ner_tags": ids})
    if unknown:
        print("Unknown tags mapped to O:", sorted(unknown))
    ds = Dataset.from_list(records)
    n_val = max(1, int(round(len(ds) * test_size)))
    split = ds.train_test_split(test_size=n_val, seed=seed, shuffle=True)
    return DatasetDict({"train": split["train"], "validation": split["test"]})


def tokenize_and_align(examples, tokenizer, max_length=256):
    tokenized = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        padding=False,
    )
    aligned = []
    for i, labels in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        previous = None
        label_ids = []
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous:
                label_ids.append(labels[word_id])
            else:
                label_ids.append(-100)  # continuation subword ignored in loss
            previous = word_id
        aligned.append(label_ids)
    tokenized["labels"] = aligned
    return tokenized


dataset = load_ner_dataset(CONLL_PATH, test_size=0.3, seed=42)
print("Labels:", LABEL_LIST)
print("Train:", len(dataset["train"]), "Val:", len(dataset["validation"]))

Labels: ['O', 'B-Product', 'I-Product', 'B-LOC', 'I-LOC', 'B-PRICE', 'I-PRICE']
Train: 70 Val: 30


In [ ]:
# 4) Fine-tune AfroXLMR with Hugging Face Trainer
import json
from functools import partial
from pathlib import Path

import numpy as np
import torch
from seqeval.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

MODEL_NAME = "xlm-roberta-base"
FALLBACK = "xlm-roberta-base"
OUTPUT_DIR = Path("/content/models/xlm-roberta-ner")
EPOCHS = 10
BATCH_SIZE = 8
LR = 3e-5
MAX_LENGTH = 256
SEED = 42

set_seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

loaded_name = None
last_err = None
for name in [MODEL_NAME, FALLBACK]:
    try:
        tokenizer = AutoTokenizer.from_pretrained(name)
        model = AutoModelForTokenClassification.from_pretrained(
            name,
            num_labels=len(LABEL_LIST),
            id2label=ID2LABEL,
            label2id=LABEL2ID,
        )
        loaded_name = name
        print("Loaded", name)
        break
    except Exception as e:
        print("Could not load", name, e)
        last_err = e
if loaded_name is None:
    raise RuntimeError(last_err)

tokenized = dataset.map(
    partial(tokenize_and_align, tokenizer=tokenizer, max_length=MAX_LENGTH),
    batched=True,
    remove_columns=["tokens", "ner_tags"],
)
collator = DataCollatorForTokenClassification(tokenizer=tokenizer)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    true_preds, true_labels = [], []
    for pred_seq, lab_seq in zip(preds, labels):
        p_out, l_out = [], []
        for p_i, l_i in zip(pred_seq, lab_seq):
            if l_i == -100:
                continue
            p_out.append(ID2LABEL[int(p_i)])
            l_out.append(ID2LABEL[int(l_i)])
        true_preds.append(p_out)
        true_labels.append(l_out)
    return {
        "precision": float(precision_score(true_labels, true_preds, zero_division=0)),
        "recall": float(recall_score(true_labels, true_preds, zero_division=0)),
        "f1": float(f1_score(true_labels, true_preds, zero_division=0)),
        "accuracy": float(accuracy_score(true_labels, true_preds)),
    }


use_fp16 = torch.cuda.is_available()

# Modern TrainingArguments directly instantiated without brittle fallback unpacking
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    warmup_steps=10,
    logging_steps=10,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=[],
    seed=SEED,
    fp16=use_fp16,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
)

train_result = trainer.train()
eval_metrics = trainer.evaluate()

best_dir = OUTPUT_DIR / "best"
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))

pred_output = trainer.predict(tokenized["validation"])
pred_ids = np.argmax(pred_output.predictions, axis=2)
true_preds, true_labels = [], []
for pred_seq, lab_seq in zip(pred_ids, pred_output.label_ids):
    p_out, l_out = [], []
    for p_i, l_i in zip(pred_seq, lab_seq):
        if l_i == -100:
            continue
        p_out.append(ID2LABEL[int(p_i)])
        l_out.append(ID2LABEL[int(l_i)])
    true_preds.append(p_out)
    true_labels.append(l_out)

report = classification_report(true_labels, true_preds, digits=4, zero_division=0)
(OUTPUT_DIR / "classification_report.txt").write_text(report, encoding="utf-8")

metrics = {
    "model_name": loaded_name,
    "n_train": len(dataset["train"]),
    "n_validation": len(dataset["validation"]),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "learning_rate": LR,
    "train_runtime_sec": float(train_result.metrics.get("train_runtime", 0.0)),
    "eval": {
        k: float(v) if isinstance(v, (int, float, np.floating)) else v
        for k, v in eval_metrics.items()
    },
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")

print("\n=== Validation (seqeval) ===")
print(report)
print(json.dumps(metrics, indent=2))

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded xlm-roberta-base


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,1.473092,0.000000,0.000000,0.000000,0.531294
2,1.730204,0.922044,0.173913,0.296296,0.219178,0.727596
3,1.125342,0.541390,0.466667,0.345679,0.397163,0.827881
4,0.646209,0.467822,0.421875,0.333333,0.372414,0.830014
5,0.411573,0.609566,0.617021,0.716049,0.662857,0.808677
6,0.313696,0.417598,0.617978,0.679012,0.647059,0.879801
7,0.226163,0.471754,0.616162,0.753086,0.677778,0.866287
8,0.153086,0.453288,0.703297,0.790123,0.744186,0.876956
9,0.151633,0.488803,0.684783,0.777778,0.728324,0.874111
10,0.112831,0.511955,0.659794,0.790123,0.719101,0.865576


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,Accuracy
0.112831,0.453288,10,0.703297,0.790123,0.744186,0.876956



=== Validation (seqeval) ===
              precision    recall  f1-score   support

         LOC     0.8929    0.8621    0.8772        29
       PRICE     0.9286    1.0000    0.9630        26
     Product     0.3714    0.5000    0.4262        26

   micro avg     0.7033    0.7901    0.7442        81
   macro avg     0.7310    0.7874    0.7555        81
weighted avg     0.7369    0.7901    0.7600        81

{
  "model_name": "xlm-roberta-base",
  "n_train": 70,
  "n_validation": 30,
  "epochs": 10,
  "batch_size": 8,
  "learning_rate": 3e-05,
  "train_runtime_sec": 547.0866,
  "eval": {
    "eval_loss": 0.45328763127326965,
    "eval_precision": 0.7032967032967034,
    "eval_recall": 0.7901234567901234,
    "eval_f1": 0.7441860465116279,
    "eval_accuracy": 0.8769559032716927
  }
}


In [23]:
!zip -r /content/xlm-roberta-ner.zip /content/models/xlm-roberta-ner/best /content/models/xlm-roberta-ner/metrics.json /content/models/xlm-roberta-ner/classification_report.txt

  adding: content/models/xlm-roberta-ner/best/ (stored 0%)
  adding: content/models/xlm-roberta-ner/best/tokenizer_config.json (deflated 49%)
  adding: content/models/xlm-roberta-ner/best/model.safetensors (deflated 27%)
  adding: content/models/xlm-roberta-ner/best/training_args.bin (deflated 53%)
  adding: content/models/xlm-roberta-ner/best/config.json (deflated 54%)
  adding: content/models/xlm-roberta-ner/best/tokenizer.json (deflated 76%)
  adding: content/models/xlm-roberta-ner/metrics.json (deflated 41%)
  adding: content/models/xlm-roberta-ner/classification_report.txt (deflated 54%)
